# 04 — Power Transition Tracker Dashboard

Builds the IEA-style HTML dashboard (`outputs/uzbekistan_power_tracker.html`) using:
- Master dataset from `01_data_pipeline`
- Forecast CSVs from `03_forecasting`
- Curated news and source documents

Each Plotly figure is exported as a stand-alone div, then assembled into a single self-contained HTML page.

In [1]:
import json, os, pathlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

ROOT = pathlib.Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
OUT  = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

pio.templates.default = 'plotly_white'

COLORS = {
    'gas':'#d97706','coal':'#374151','hydro':'#0891b2','solar':'#facc15',
    'wind':'#10b981','demand':'#1e3a8a','fossil':'#7f1d1d','renewable':'#16a34a',
    'co2':'#b91c1c','BAU':'#9ca3af','Government':'#16a34a','Accelerated':'#0d9488',
    'history':'#1f2937','ensemble':'#111827',
}

In [2]:
df = pd.read_csv(DATA / 'master_dataset_core.csv')
df_c = df[df['data_status']=='confirmed'].copy()
df_p = df[df['data_status']=='preliminary'].copy()

demand_fc    = pd.read_csv(DATA / 'forecast_demand.csv')
scenarios_fc = pd.read_csv(DATA / 'forecast_scenarios.csv')
co2_fc       = pd.read_csv(DATA / 'forecast_co2.csv')
invest_fc    = pd.read_csv(DATA / 'investment_signals.csv')

print(f'Historical: {df_c.year.min()}–{df_c.year.max()}')
print(f'Forecast horizon: {demand_fc.year.min()}–{demand_fc.year.max()}')

Historical: 1990–2023
Forecast horizon: 2024–2040


## Figure factory — one builder per energy type

In [3]:
def to_div(fig, div_id, height=420):
    fig.update_layout(height=height, margin=dict(l=50, r=20, t=50, b=40),
                      font=dict(family='-apple-system, system-ui, sans-serif', size=12),
                      legend=dict(orientation='h', yanchor='bottom', y=-0.25,
                                   xanchor='left', x=0))
    return pio.to_html(fig, include_plotlyjs=False, full_html=False, div_id=div_id,
                       config={'displayModeBar': False, 'responsive': True})

def hist_line(df_, x, y, name, color, dash=None, mode='lines+markers'):
    return go.Scatter(x=df_[x], y=df_[y], name=name, mode=mode,
                       line=dict(color=color, width=2, dash=dash),
                       marker=dict(size=5))

### Overview — generation mix + KPI

In [4]:
# ── Stacked area: generation mix history + Government scenario forward ──
hist_gen = df_c[['year','gen_gas_twh','gen_coal_twh','gen_hydro_twh','gen_solar_twh','gen_wind_twh']].copy()
for c in hist_gen.columns[1:]:
    hist_gen[c] = hist_gen[c].fillna(0)

govt_fc = scenarios_fc[scenarios_fc['scenario']=='Government'].copy()
# Split thermal into gas/coal at 88/12
govt_fc['gen_gas_twh']  = govt_fc['gen_thermal_twh'] * 0.88
govt_fc['gen_coal_twh'] = govt_fc['gen_thermal_twh'] * 0.12

fig = go.Figure()
for src, col in [('gen_gas_twh','gas'),('gen_coal_twh','coal'),
                 ('gen_hydro_twh','hydro'),('gen_solar_twh','solar'),('gen_wind_twh','wind')]:
    fig.add_trace(go.Scatter(
        x=pd.concat([hist_gen['year'], govt_fc['year']]),
        y=pd.concat([hist_gen[src],    govt_fc[src]]),
        name=col.title(), stackgroup='one', mode='none',
        fillcolor=COLORS[col], line=dict(width=0),
        hovertemplate=f'<b>{col.title()}</b>: %{{y:.1f}} TWh<extra></extra>'))
fig.add_vline(x=2023.5, line=dict(dash='dot', color='grey'),
              annotation_text='forecast →', annotation_position='top right')
fig.update_layout(title='Electricity Generation by Source — History + Government Target Scenario',
                  yaxis_title='TWh', xaxis_title='Year', hovermode='x unified')
fig_overview = to_div(fig, 'overview-mix', 460)
fig.show()

In [5]:
# ── RE share scenarios ──
fig = go.Figure()
re_hist = df_c[['year','re_penetration_pct']].dropna()
fig.add_trace(hist_line(re_hist, 'year','re_penetration_pct',
                         'Historical', COLORS['history']))
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['re_share_pct'], name=f'{sc} scenario',
                              line=dict(color=COLORS[sc], width=2.5, dash='dash')))
fig.add_hline(y=54, line=dict(dash='dot', color='grey'),
              annotation_text='Strategy 2030 target (54%)', annotation_position='top left')
fig.add_vline(x=2023.5, line=dict(dash='dot', color='grey'))
fig.update_layout(title='Renewable Share of Generation (%) — Three Scenarios',
                  yaxis_title='% RE', xaxis_title='Year', hovermode='x unified')
fig_re = to_div(fig, 'overview-re', 420)
fig.show()

In [6]:
# ── Demand forecast: winner + 80%/95% CIs + alternative models for context ──
import warnings; warnings.filterwarnings('ignore')
fig = go.Figure()
y_h = df_c[['year','elec_consumption_twh_bridged']].dropna()
fig.add_trace(hist_line(y_h, 'year','elec_consumption_twh_bridged',
                         'History (consumption)', COLORS['history']))
fig.add_trace(go.Scatter(x=demand_fc['year'], y=demand_fc['ci_hi95'],
                          mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=demand_fc['year'], y=demand_fc['ci_lo95'],
                          mode='lines', fill='tonexty', line=dict(width=0),
                          fillcolor='rgba(30,58,138,0.08)', name='95% CI'))
fig.add_trace(go.Scatter(x=demand_fc['year'], y=demand_fc['ci_hi80'],
                          mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=demand_fc['year'], y=demand_fc['ci_lo80'],
                          mode='lines', fill='tonexty', line=dict(width=0),
                          fillcolor='rgba(30,58,138,0.18)', name='80% CI'))
winner_lbl = demand_fc['winner_model'].iat[0]
fig.add_trace(go.Scatter(x=demand_fc['year'], y=demand_fc['demand_twh'],
                          name=f'★ {winner_lbl}', line=dict(color='#111827', width=3)))
fig.add_vline(x=2023.5, line=dict(dash='dot', color='grey'))
fig.add_trace(go.Scatter(x=[2025,2026], y=[86.7/1.09, 90.0/1.09],
                          mode='markers', name='News-reported (gen/1.09)',
                          marker=dict(symbol='star', size=14, color='#dc2626')))
fig.update_layout(title=f'Electricity Demand Forecast 2024–2040 — winner: {winner_lbl}',
                  yaxis_title='TWh', xaxis_title='Year', hovermode='x unified')
fig_demand = to_div(fig, 'demand-forecast', 460)
fig.show()

In [7]:
# ── Gas: generation + self-sufficiency ──
fig = make_subplots(rows=1, cols=2, subplot_titles=('Gas-fired generation', 'Gas self-sufficiency (production/consumption)'))
gh = df_c[['year','gen_gas_twh']].dropna()
fig.add_trace(hist_line(gh,'year','gen_gas_twh','Gas gen (TWh)', COLORS['gas']), row=1, col=1)
# Forecasts: thermal × 0.88 ≈ gas
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['gen_thermal_twh']*0.88,
                              name=f'Gas — {sc}',
                              line=dict(color=COLORS[sc], dash='dash', width=2)), row=1, col=1)
gs = df[['year','gas_self_sufficiency_pct']].dropna()
fig.add_trace(go.Scatter(x=gs['year'], y=gs['gas_self_sufficiency_pct'],
                          name='Self-sufficiency %', line=dict(color=COLORS['gas'], width=2),
                          mode='lines+markers', marker=dict(size=5)), row=1, col=2)
fig.add_hline(y=100, line=dict(dash='dot', color='grey'), row=1, col=2)
fig.update_yaxes(title_text='TWh', row=1, col=1)
fig.update_yaxes(title_text='%', row=1, col=2)
fig.update_layout(title='Gas — Generation & Self-Sufficiency', hovermode='x unified')
fig_gas = to_div(fig, 'gas-panel', 440)
fig.show()

In [8]:
fig = go.Figure()
hh = df_c[['year','gen_hydro_twh']].dropna()
fig.add_trace(hist_line(hh,'year','gen_hydro_twh','Historical hydro gen (TWh)', COLORS['hydro']))
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['gen_hydro_twh'], name=f'{sc}',
                              line=dict(color=COLORS[sc], dash='dash', width=2)))
fig.add_vline(x=2023.5, line=dict(dash='dot', color='grey'))
fig.update_layout(title='Hydropower Generation — Historical + Scenarios',
                  yaxis_title='TWh', xaxis_title='Year', hovermode='x unified')
fig_hydro = to_div(fig, 'hydro-panel', 420)
fig.show()

In [9]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Solar generation', 'Solar installed capacity'))
sh = df_c[['year','gen_solar_twh']].fillna(0)
fig.add_trace(go.Bar(x=sh['year'], y=sh['gen_solar_twh'], name='Historical (TWh)',
                      marker_color=COLORS['solar']), row=1, col=1)
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['gen_solar_twh'], name=f'{sc}',
                              line=dict(color=COLORS[sc], dash='dash', width=2)), row=1, col=1)
# Capacity
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['cap_solar_mw']/1000, name=f'{sc} cap',
                              line=dict(color=COLORS[sc], dash='dot', width=2), showlegend=False), row=1, col=2)
fig.update_yaxes(title_text='TWh', row=1, col=1)
fig.update_yaxes(title_text='GW', row=1, col=2)
fig.update_layout(title='Solar PV — Generation & Capacity', hovermode='x unified')
fig_solar = to_div(fig, 'solar-panel', 440)
fig.show()

In [10]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Wind generation','Wind installed capacity'))
wh = df_c[['year','gen_wind_twh']].fillna(0)
fig.add_trace(go.Bar(x=wh['year'], y=wh['gen_wind_twh'], name='Historical (TWh)',
                      marker_color=COLORS['wind']), row=1, col=1)
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['gen_wind_twh'], name=f'{sc}',
                              line=dict(color=COLORS[sc], dash='dash', width=2)), row=1, col=1)
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['cap_wind_mw']/1000, name=f'{sc} cap',
                              line=dict(color=COLORS[sc], dash='dot', width=2), showlegend=False), row=1, col=2)
fig.update_yaxes(title_text='TWh', row=1, col=1)
fig.update_yaxes(title_text='GW', row=1, col=2)
fig.update_layout(title='Wind — Generation & Capacity', hovermode='x unified')
fig_wind = to_div(fig, 'wind-panel', 440)
fig.show()

In [11]:
fig = go.Figure()
ch = df_c[['year','gen_coal_twh']].dropna()
fig.add_trace(hist_line(ch,'year','gen_coal_twh','Historical coal gen', COLORS['coal']))
for sc in ['BAU','Government','Accelerated']:
    sd = scenarios_fc[scenarios_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['gen_thermal_twh']*0.12,
                              name=f'Coal — {sc} (12% of thermal)',
                              line=dict(color=COLORS[sc], dash='dash', width=2)))
fig.add_vline(x=2023.5, line=dict(dash='dot', color='grey'))
fig.update_layout(title='Coal Generation — Historical + Scenarios',
                  yaxis_title='TWh', xaxis_title='Year', hovermode='x unified')
fig_coal = to_div(fig, 'coal-panel', 420)
fig.show()

In [12]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Power-sector CO₂ emissions','CO₂ intensity'))
ch = df_c[['year','wb_co2_power_mt']].dropna()
fig.add_trace(hist_line(ch,'year','wb_co2_power_mt','History (WB)', COLORS['history']), row=1, col=1)
for sc in ['BAU','Government','Accelerated']:
    sd = co2_fc[co2_fc['scenario']==sc]
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['co2_power_mt'], name=f'{sc}',
                              line=dict(color=COLORS[sc], dash='dash', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=sd['year'], y=sd['co2_intensity_gco2_per_kwh'],
                              name=f'{sc} (intensity)',
                              line=dict(color=COLORS[sc], dash='dash', width=2), showlegend=False),
                              row=1, col=2)
# Historical intensity
ih = df_c[['year','wb_co2_power_mt','gen_total_twh_bridged']].dropna()
ih['intensity'] = ih['wb_co2_power_mt']*1e3 / ih['gen_total_twh_bridged']
fig.add_trace(hist_line(ih,'year','intensity','History', COLORS['history']), row=1, col=2)
fig.add_hline(y=475, line=dict(dash='dot', color='grey'), row=1, col=2,
              annotation_text='World avg', annotation_position='top right')
fig.update_yaxes(title_text='Mt CO₂/yr', row=1, col=1)
fig.update_yaxes(title_text='gCO₂/kWh', row=1, col=2)
fig.update_layout(title='Power-sector Carbon Footprint', hovermode='x unified')
fig_co2 = to_div(fig, 'co2-panel', 440)
fig.show()

In [13]:
pivot = invest_fc.pivot(index='tech', columns='scenario', values='capex_bn_usd')
pivot = pivot[['BAU','Government','Accelerated']].reindex(['solar','wind','hydro','thermal'])
fig = go.Figure()
for sc in pivot.columns:
    fig.add_trace(go.Bar(name=sc, x=pivot.index, y=pivot[sc],
                         marker_color=COLORS[sc],
                         text=[f'${v:.1f} bn' for v in pivot[sc]], textposition='outside'))
fig.update_layout(barmode='group',
                  title='Implied Capex by Technology & Scenario, 2024–2040 (USD bn)',
                  yaxis_title='USD billion', xaxis_title='Technology', hovermode='x unified')
fig_invest = to_div(fig, 'invest-panel', 440)
fig.show()

In [14]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Installed capacity (historical)','Capacity outlook — Govt Target scenario'))
ch = df[['year','capacity_thermal_mw','capacity_hydro_mw']].dropna()
fig.add_trace(go.Scatter(x=ch['year'], y=ch['capacity_thermal_mw']/1000, stackgroup='hist',
                          name='Thermal (hist)', fillcolor=COLORS['gas'], line=dict(width=0)), row=1, col=1)
fig.add_trace(go.Scatter(x=ch['year'], y=ch['capacity_hydro_mw']/1000, stackgroup='hist',
                          name='Hydro (hist)', fillcolor=COLORS['hydro'], line=dict(width=0)), row=1, col=1)
sd = scenarios_fc[scenarios_fc['scenario']=='Government']
for col, name in [('cap_thermal_mw','Thermal'),('cap_hydro_mw','Hydro'),
                  ('cap_solar_mw','Solar'),('cap_wind_mw','Wind')]:
    fig.add_trace(go.Scatter(x=sd['year'], y=sd[col]/1000, stackgroup='fc',
                              name=f'{name} (fc)', fillcolor=COLORS[{'thermal':'gas','hydro':'hydro','solar':'solar','wind':'wind'}[name.lower()]], line=dict(width=0)),
                              row=1, col=2)
fig.update_yaxes(title_text='GW', row=1, col=1); fig.update_yaxes(title_text='GW', row=1, col=2)
fig.update_layout(title='Installed Capacity — History & Government Target', hovermode='x unified')
fig_cap = to_div(fig, 'cap-panel', 440)
fig.show()

## Assemble HTML dashboard

Single-page layout with sticky left navigation. All charts and data are self-contained — no external dependencies except CDN-loaded `plotly.min.js`.

In [15]:
# Key KPIs for the hero section
latest = df_c[df_c['year']==df_c['year'].max()].iloc[0]
kpi_demand_2030 = demand_fc.loc[demand_fc['year']==2030, 'demand_twh'].iat[0]
kpi_re_2030     = scenarios_fc[(scenarios_fc['scenario']=='Government') & (scenarios_fc['year']==2030)]['re_share_pct'].iat[0]
kpi_co2_2030    = co2_fc[(co2_fc['scenario']=='Government') & (co2_fc['year']==2030)]['co2_power_mt'].iat[0]
kpi_capex_govt  = invest_fc[invest_fc['scenario']=='Government']['capex_bn_usd'].sum()
winner_lbl = demand_fc['winner_model'].iat[0]
kpi = dict(
    latest_year = int(latest['year']),
    demand_now  = round(latest['elec_consumption_twh_bridged'], 1),
    demand_2030 = round(kpi_demand_2030, 1),
    re_now      = round(latest['re_penetration_pct'], 1),
    re_2030     = round(kpi_re_2030, 1),
    co2_now     = round(latest['wb_co2_power_mt'], 1),
    co2_2030    = round(kpi_co2_2030, 1),
    capex_govt  = round(kpi_capex_govt, 1),
    winner      = winner_lbl,
)
kpi

{'latest_year': 2023,
 'demand_now': np.float64(73.4),
 'demand_2030': np.float64(103.3),
 're_now': np.float64(10.2),
 're_2030': np.float64(48.6),
 'co2_now': np.float64(47.7),
 'co2_2030': np.float64(34.0),
 'capex_govt': np.float64(56.9),
 'winner': 'Prophet'}

In [16]:
# Curated news (May 2025 onward) — pulled via web search at notebook-build time
news_items = [
    {'date':'May 2026', 'title':'Uzbekistan Energy Week 2026 — record participation',
     'summary':'580 companies, 140+ speakers, 1,500 delegates from 34 countries — largest UEW edition ever.',
     'url':'https://dunyo.info/en/reportazh-ia-dunyo/energeticheskaya-nedelya-uzbekistana-uew-2026-sobrala-rekordnoe-chislo',
     'source':'Dunyo'},
    {'date':'May 2026', 'title':'ACWA Power secures USD 226 M for Bash II 300 MW wind farm',
     'summary':'Financing closed at ADB Annual Meeting with ADB, AIIB and Standard Chartered — adds to ACWA Power\'s rapid Uzbek pipeline.',
     'url':'https://solarquarter.com/2026/05/11/acwa-power-secures-usd-226-million-international-financing-to-accelerate-uzbekistans-renewable-energy-and-energy-transition-goals/',
     'source':'SolarQuarter'},
    {'date':'Apr 2026', 'title':'Solar & wind plants pass 1 billion kWh of green generation YTD',
     'summary':'Combined RE generation Jan–Apr 2026: 1.0 TWh — saving 267 M m³ natural gas and avoiding 575 kt of emissions.',
     'url':'https://www.uzdaily.uz/en/solar-and-wind-power-plants-in-uzbekistan-generate-1-billion-kwh-of-green-energy-in-2026/',
     'source':'UzDaily'},
    {'date':'Jan 2026', 'title':'Masdar reaches financial close on 300 MW Guzar PV + 75 MWh BESS',
     'summary':'ADB-Masdar $30 M deal with 1.6 km transmission tie-in to 220 kV substation in Kashkadarya region.',
     'url':'https://www.adb.org/news/adb-masdar-sign-30-million-deal-boost-solar-energy-bess-capacity-uzbekistan',
     'source':'ADB'},
    {'date':'Dec 2025', 'title':'Uzbekistan plans 6.7 GW of new capacity in 2026',
     'summary':'Additions: 2.8 GW solar, 2.5 GW thermal, 884 MW storage, 470 MW wind, 68 MW hydro. Generation rises 3.8% to 90 TWh.',
     'url':'https://www.enerdata.net/publications/daily-energy-news/uzbekistan-plans-67-gw-new-capacity-including-25-gw-thermal-2026.html',
     'source':'Enerdata'},
    {'date':'Dec 2025', 'title':'EBRD invests USD 230 M in ACWA Power solar+storage',
     'summary':'Financing for ACWA\'s Uzbek solar-plus-storage portfolio; reflects record EBRD commitment to Uzbek energy transition.',
     'url':'https://www.pv-tech.org/ebrd-invests-us230-million-in-acwa-powers-uzbekistan-solar-plus-storage-project/',
     'source':'PV-Tech'},
    {'date':'Apr 2025', 'title':'President: RE share to reach 54% by 2030',
     'summary':'Targets revised from earlier 40% to 54% RE share of generation; 17 GW additional RE by 2030.',
     'url':'https://kun.uz/en/news/2025/04/19/uzbekistan-aims-to-generate-54-of-electricity-from-renewable-sources-by-2030',
     'source':'Kun.uz'},
    {'date':'Apr 2024', 'title':'Electricity & gas tariffs raised — first cost-reflective step',
     'summary':'Residential rate 450 UZS/kWh up to 200 kWh; 900 UZS/kWh above. Full gas price deregulation planned 2026.',
     'url':'https://www.gazeta.uz/en/2024/04/16/tariffs/',
     'source':'Gazeta.uz'},
]

documents = [
    {'title':'EDB 2026 — Power Sector of Central Asia (regional report)',
     'desc':'Eurasian Development Bank flagship report on power-sector outlook for Central Asia, 2026 edition.',
     'href':'../readings/EDB_2026_Power-sector-of-CA_Report_EN.pdf',
     'tag':'Industry report'},
    {'title':'White Paper — Power Sector Development in Central Asia',
     'desc':'Regional energy transition white paper covering capacity, transmission, and investment frameworks.',
     'href':'../readings/White Paper_Power_Sector_Development_in_Central_Asia.pdf',
     'tag':'White paper'},
    {'title':'IEA — Solar Energy Policy in Uzbekistan: A Roadmap',
     'desc':'IEA roadmap for solar deployment in Uzbekistan to 2030; policy & regulatory analysis.',
     'href':'https://www.iea.org/reports/solar-energy-policy-in-uzbekistan-a-roadmap',
     'tag':'IEA report'},
    {'title':'IEA — Uzbekistan Energy Profile (Efficiency & Demand)',
     'desc':'IEA country page used as the visual template for this tracker.',
     'href':'https://www.iea.org/countries/uzbekistan/efficiency-demand',
     'tag':'IEA country page'},
    {'title':'Climatescope 2025 — Uzbekistan',
     'desc':'BloombergNEF Climatescope investment-attractiveness rating for Uzbekistan, 2025.',
     'href':'https://www.global-climatescope.org/markets/uzbekistan',
     'tag':'Investment'},
    {'title':'Project Initiation Document — Farangiz Jurakhonova (capstone)',
     'desc':'Capstone PID setting out terms, scope and milestones with ILF Consulting.',
     'href':'../readings/Project Initiation Document _Farangiz Jurakhonova.docx',
     'tag':'Capstone'},
    {'title':'ILF Capstone topic — Power Sector Transition Tracker',
     'desc':'Original brief from ILF Consulting for the capstone project.',
     'href':'../readings/Capstone topic ILF Ardak.docx',
     'tag':'Brief'},
    {'title':'Data sources catalogue',
     'desc':'Links to every source used to build master_dataset.csv (IEA, IRENA, WB, StatSUZ).',
     'href':'../readings/Data sources (links).docx',
     'tag':'Data'},
]
len(news_items), len(documents)

(8, 8)

In [17]:
# Build the final HTML page
import html as html_lib
def esc(s): return html_lib.escape(str(s))


# Glossary content
GLOSSARY = [
    ('Units', [
        ('TWh', 'Terawatt-hour. 1 TWh = 1 billion kWh = power for ~100,000 average households for a year.'),
        ('GW / MW', 'Gigawatt / Megawatt. Units of capacity (power). 1 GW = 1,000 MW. A gas plant is 200–800 MW; a wind turbine 3–6 MW.'),
        ('bcm', 'Billion cubic metres — gas volume. Uzbekistan consumed ~52 bcm in 2023; ~17 bcm went into electricity.'),
        ('gCO₂/kWh', 'Grams of CO₂ per kWh generated. World avg ~475; gas CCGT ~380; coal ~950; renewables ~0.'),
    ]),
    ('Scenarios', [
        ('BAU', 'Business-As-Usual — current build pace continues; reaches ~60% of the 2030 RE target.'),
        ('Government Target', 'Meets official April 2025 targets exactly: 12 GW solar, 8 GW wind, 4.7 GW hydro by 2030.'),
        ('Accelerated', 'Stretch case — exceeds government targets; reaches 60%+ RE share by 2030.'),
        ('Plan B (Nuclear)', 'Parametric sensitivity — toggles a 1.2–3.6 GW SMR commissioned 2030–2034. Not a baseline forecast.'),
    ]),
    ('Power-sector concepts', [
        ('RE share', 'Share of generation from hydro+solar+wind. Uzbekistan ~10% in 2023; target 54% by 2030.'),
        ('Capacity factor (CF)', 'How much of nameplate capacity a plant produces yearly. Solar 18%, wind 30%, hydro 36%, gas 55%, nuclear 85%.'),
        ('T&D losses', 'Transmission & Distribution losses. Uzbekistan ~9–10%; regional median ~8%. A grid-investment signal.'),
        ('Gas self-sufficiency', 'Domestic production ÷ consumption. Below 100% = net importer (winter shortages since 2018).'),
    ]),
    ('Forecasting terms', [
        ('MAPE', 'Mean Absolute Percentage Error. Lewis (1982): <10% "highly accurate", 10–20% "good".'),
        ('Cross-validation (CV)', '8 expanding-window splits × 3-year horizon — fair accuracy test on past data.'),
        ('Bootstrap CI', 'Confidence band from resampling residuals 1,000×. 80% CI = expect truth inside 80% of the time.'),
        ('Hold-out test', 'A chunk of recent data (2019–2023) reserved — not used for training.'),
    ]),
    ('Models', [
        ('Prophet', 'Meta\'s time-series tool. Trend + seasonality + changepoints. Won our bench (CV MAPE 6.4%).'),
        ('ARIMA / SARIMAX', 'Classical statistical TS models; predict next year from past. SARIMAX adds external vars (GDP, pop).'),
        ('Holt-Winters', 'Exponential smoothing. Robust on small samples; placed 2nd in our bench.'),
        ('Theta method', 'Decompose-and-recombine — M3-competition classic.'),
        ('OLS first-difference', 'Linear regression on year-on-year changes; avoids spurious regression (Granger-Newbold 1974).'),
        ('Ridge / Gradient Boosting', 'ML models on lagged + macro features; overfit on short annual series.'),
    ]),
    ('Institutions', [
        ('IEA', 'International Energy Agency (Paris).'),
        ('IRENA', 'International Renewable Energy Agency (Abu Dhabi).'),
        ('WB', 'World Bank — Data360 for GDP, population, CO₂.'),
        ('EBRD / ADB / AIIB / IFC', 'Multilateral financiers active in Uzbek power.'),
        ('NEEA', 'National Energy Efficiency Agency of Uzbekistan — ILF\'s key counterpart.'),
        ('StatSUZ', 'State Statistics Committee of Uzbekistan.'),
    ]),
    ('Technologies', [
        ('CCGT / OCGT', 'Combined-Cycle / Open-Cycle Gas Turbine. CCGT modern & efficient; OCGT older. UZ fleet mostly OCGT/CHP today.'),
        ('PV', 'Photovoltaic — solar from semiconductor panels.'),
        ('BESS', 'Battery Energy Storage System. ~5 GWh needed by 2030 (Govt scenario).'),
        ('SMR', 'Small Modular Reactor — the nuclear tech for Jizzakh (Plan B).'),
        ('PPP / IPP', 'Public-Private Partnership / Independent Power Producer — ACWA, Masdar, EDF deal structures.'),
        ('PPA', 'Power Purchase Agreement — fixed long-term offtake contract.'),
    ]),
]

glossary_html = ''
for cat, items in GLOSSARY:
    inner = ''
    for term, defn in items:
        inner += f'<div style="margin-bottom:8px;font-size:13px;"><span style="font-weight:600;color:#0d4d7a;">{esc(term)}</span> — <span style="color:#374151;">{esc(defn)}</span></div>'
    glossary_html += f'<div class="card"><h3 style="margin:0 0 10px;font-size:15px;">{esc(cat)}</h3>{inner}</div>'

news_html = ''
for n in news_items:
    news_html += (
        f'<article class="news-item">'
        f'<div class="news-date">{esc(n["date"])}</div>'
        f'<a class="news-title" href="{esc(n["url"])}" target="_blank" rel="noopener">{esc(n["title"])}</a>'
        f'<p class="news-summary">{esc(n["summary"])}</p>'
        f'<div class="news-source">— {esc(n["source"])}</div>'
        f'</article>'
    )

docs_html = ''
for d in documents:
    docs_html += (
        f'<a class="doc-card" href="{esc(d["href"])}" target="_blank" rel="noopener">'
        f'<div class="doc-tag">{esc(d["tag"])}</div>'
        f'<div class="doc-title">{esc(d["title"])}</div>'
        f'<p class="doc-desc">{esc(d["desc"])}</p>'
        f'</a>'
    )

kpi_cards = f"""
<div class="kpi-grid">
  <div class="kpi"><span class="kpi-label">Demand {kpi['latest_year']}</span><span class="kpi-value">{kpi['demand_now']} <small>TWh</small></span><span class="kpi-sub">→ {kpi['demand_2030']} TWh by 2030 (ensemble)</span></div>
  <div class="kpi"><span class="kpi-label">RE share {kpi['latest_year']}</span><span class="kpi-value">{kpi['re_now']}<small>%</small></span><span class="kpi-sub">→ {kpi['re_2030']}% in 2030 (Govt target)</span></div>
  <div class="kpi"><span class="kpi-label">Power CO₂ {kpi['latest_year']}</span><span class="kpi-value">{kpi['co2_now']} <small>Mt</small></span><span class="kpi-sub">→ {kpi['co2_2030']} Mt in 2030 (Govt)</span></div>
  <div class="kpi"><span class="kpi-label">Capex 2024-35</span><span class="kpi-value">${kpi['capex_govt']}<small> bn</small></span><span class="kpi-sub">Government scenario, all techs</span></div>
</div>
"""

page = f"""<!doctype html>
<html lang='en'>
<head>
<meta charset='utf-8'/>
<meta name='viewport' content='width=device-width,initial-scale=1'/>
<title>Uzbekistan — Power Sector Transition Tracker</title>
<script src='https://cdn.plot.ly/plotly-2.35.2.min.js'></script>
<style>
  :root {{
    --bg:#fafafa; --card:#ffffff; --ink:#111827; --muted:#6b7280;
    --line:#e5e7eb; --accent:#0d4d7a; --accent-light:#dbeafe;
    --gas:#d97706; --hydro:#0891b2; --solar:#facc15; --wind:#10b981; --coal:#374151;
  }}
  * {{ box-sizing: border-box; }}
  body {{ font-family:-apple-system,system-ui,'Segoe UI',Roboto,sans-serif;
         background:var(--bg); color:var(--ink); margin:0; line-height:1.45; }}
  header.banner {{ background:linear-gradient(135deg,#0d4d7a 0%,#1e3a8a 100%);
                   color:white; padding:28px 40px; }}
  header.banner h1 {{ margin:0 0 6px; font-size:26px; letter-spacing:-0.2px; }}
  header.banner p  {{ margin:0; opacity:0.85; font-size:14px; }}
  .layout {{ display:grid; grid-template-columns:240px 1fr; gap:0; }}
  nav.side {{ background:white; border-right:1px solid var(--line);
              padding:24px 0; position:sticky; top:0; height:100vh; overflow-y:auto; }}
  nav.side ul {{ list-style:none; padding:0; margin:0; }}
  nav.side a {{ display:block; padding:9px 20px; color:var(--ink);
                 text-decoration:none; font-size:14px; border-left:3px solid transparent; }}
  nav.side a:hover {{ background:var(--accent-light); border-left-color:var(--accent); }}
  nav.side h4 {{ margin:18px 20px 6px; color:var(--muted); font-size:11px;
                  text-transform:uppercase; letter-spacing:1px; font-weight:600; }}
  main {{ padding:32px 40px 80px; max-width:1100px; }}
  section.tab {{ margin-bottom:48px; scroll-margin-top:20px; }}
  section.tab h2 {{ font-size:22px; margin:0 0 4px; }}
  section.tab .lede {{ color:var(--muted); margin:0 0 18px; font-size:14px; }}
  .card {{ background:white; border:1px solid var(--line); border-radius:10px;
           padding:20px; margin-bottom:20px; box-shadow:0 1px 2px rgba(0,0,0,0.02); }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr));
                gap:14px; margin-bottom:24px; }}
  .kpi {{ background:white; border:1px solid var(--line); border-radius:10px;
          padding:18px; display:flex; flex-direction:column; gap:4px; }}
  .kpi-label {{ font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }}
  .kpi-value {{ font-size:28px; font-weight:600; color:var(--accent); }}
  .kpi-value small {{ font-size:14px; color:var(--muted); font-weight:400; margin-left:2px; }}
  .kpi-sub {{ font-size:12px; color:var(--muted); }}
  .news-item {{ padding:14px 0; border-bottom:1px solid var(--line); }}
  .news-item:last-child {{ border-bottom:0; }}
  .news-date {{ color:var(--muted); font-size:11px; text-transform:uppercase; letter-spacing:.5px; }}
  .news-title {{ display:block; font-size:16px; font-weight:600; color:var(--ink);
                  text-decoration:none; margin:3px 0; }}
  .news-title:hover {{ color:var(--accent); text-decoration:underline; }}
  .news-summary {{ margin:4px 0; color:var(--ink); font-size:14px; }}
  .news-source {{ color:var(--muted); font-size:12px; font-style:italic; }}
  .docs-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:14px; }}
  .doc-card {{ background:white; border:1px solid var(--line); border-radius:8px;
                padding:16px; text-decoration:none; color:var(--ink);
                transition:border-color .15s, transform .15s; display:block; }}
  .doc-card:hover {{ border-color:var(--accent); transform:translateY(-1px); }}
  .doc-tag {{ font-size:10px; text-transform:uppercase; color:var(--accent);
              letter-spacing:1px; font-weight:600; margin-bottom:6px; }}
  .doc-title {{ font-weight:600; font-size:15px; margin-bottom:6px; }}
  .doc-desc {{ font-size:13px; color:var(--muted); margin:0; line-height:1.4; }}
  footer {{ padding:30px 40px; color:var(--muted); font-size:12px; border-top:1px solid var(--line); }}
  .pill {{ display:inline-block; padding:2px 8px; border-radius:10px;
           font-size:11px; font-weight:600; margin-right:4px; }}
  .pill-gas {{ background:#fed7aa; color:#9a3412; }}
  .pill-hydro {{ background:#a5f3fc; color:#0e7490; }}
  .pill-solar {{ background:#fef08a; color:#854d0e; }}
  .pill-wind {{ background:#bbf7d0; color:#14532d; }}
  .pill-coal {{ background:#d1d5db; color:#1f2937; }}
</style>
</head>
<body>
<header class='banner'>
  <h1>🇺🇿 Uzbekistan — Power Sector Transition Tracker</h1>
  <p>Capstone Project • Farangiz Jurakhonova • Central European University × ILF Consulting Engineers</p>
  <p style='font-size:12px;margin-top:6px;opacity:.7'>Historical data: 1990–2024 | Forecast horizon: 2024–2040 | Last refresh: May 2026</p>
</header>
<div class='layout'>
<nav class='side'>
  <h4>Overview</h4>
  <ul>
    <li><a href='#overview'>Country snapshot</a></li>
    <li><a href='#demand'>Electricity demand</a></li>
  </ul>
  <h4>By energy source</h4>
  <ul>
    <li><a href='#gas'><span class='pill pill-gas'>●</span>Natural gas</a></li>
    <li><a href='#hydro'><span class='pill pill-hydro'>●</span>Hydropower</a></li>
    <li><a href='#solar'><span class='pill pill-solar'>●</span>Solar PV</a></li>
    <li><a href='#wind'><span class='pill pill-wind'>●</span>Wind</a></li>
    <li><a href='#coal'><span class='pill pill-coal'>●</span>Coal</a></li>
    <li><a href='#capacity'>Installed capacity</a></li>
  </ul>
  <h4>Transition signals</h4>
  <ul>
    <li><a href='#co2'>CO₂ emissions</a></li>
    <li><a href='#invest'>Investment outlook</a></li>
  </ul>
  <h4>Resources</h4>
  <ul>
    <li><a href='#docs'>Documents</a></li>
    <li><a href='#news'>News</a></li>
    <li><a href='#methodology'>Methodology</a></li>
    <li><a href='#glossary'>Glossary</a></li>
  </ul>
</nav>
<main>
  <section id='overview' class='tab'>
    <h2>Country snapshot</h2>
    <p class='lede'>Headline indicators for {kpi['latest_year']} with forecast endpoints under the Government Target scenario.</p>
    {kpi_cards}
    <div class='card'>{fig_overview}</div>
    <div class='card'>{fig_re}</div>
  </section>

  <section id='demand' class='tab'>
    <h2>Electricity demand outlook</h2>
    <p class='lede'>Ensemble of ARIMA, Holt-Winters, and a GDP-elasticity regression. ARIMA 80% confidence band shaded. Red stars are news-reported actuals not yet in the dataset.</p>
    <div class='card'>{fig_demand}</div>
  </section>

  <section id='gas' class='tab'>
    <h2>Natural gas</h2>
    <p class='lede'>Gas is ~76% of generation. Self-sufficiency has fallen below 100% — Uzbekistan is now a net gas importer in winter.</p>
    <div class='card'>{fig_gas}</div>
  </section>

  <section id='hydro' class='tab'>
    <h2>Hydropower</h2>
    <p class='lede'>Capacity expansion from 2.05 GW (2023) to 4.7 GW (2030 target). Output highly weather-dependent.</p>
    <div class='card'>{fig_hydro}</div>
  </section>

  <section id='solar' class='tab'>
    <h2>Solar PV</h2>
    <p class='lede'>From near-zero in 2020 to ~5.2 GW by end-2025 — fastest-growing RE technology. Target: 12 GW by 2030.</p>
    <div class='card'>{fig_solar}</div>
  </section>

  <section id='wind' class='tab'>
    <h2>Wind</h2>
    <p class='lede'>First utility-scale wind commissioned 2024. Karakalpakstan & Bukhara are key sites. Target: 8 GW by 2030.</p>
    <div class='card'>{fig_wind}</div>
  </section>

  <section id='coal' class='tab'>
    <h2>Coal</h2>
    <p class='lede'>Small share (~8 TWh) at the Angren plant. No phase-out scheduled but plateau in all scenarios.</p>
    <div class='card'>{fig_coal}</div>
  </section>

  <section id='capacity' class='tab'>
    <h2>Installed capacity</h2>
    <p class='lede'>Total capacity must roughly double by 2030 to meet demand + reserve. Mix shifts decisively toward solar+wind.</p>
    <div class='card'>{fig_cap}</div>
  </section>

  <section id='co2' class='tab'>
    <h2>Power-sector CO₂</h2>
    <p class='lede'>Government scenario brings emissions down ~20% by 2030 via RE growth + gas-fleet modernisation (older OCGTs → CCGTs).</p>
    <div class='card'>{fig_co2}</div>
  </section>

  <section id='invest' class='tab'>
    <h2>Investment outlook</h2>
    <p class='lede'>Implied capital expenditure 2024-2040, using IRENA/IEA unit-cost ranges (solar $950/kW, wind $1,450/kW, hydro $2,200/kW, CCGT $1,100/kW).</p>
    <div class='card'>{fig_invest}</div>
    <div class='card'>
    <h3 style='margin-top:0'>Investment signals for ILF</h3>
    <ul>
      <li><strong>Solar PV + storage</strong> is the largest line item — $10 bn under Government scenario. PPP/IPP pipeline dominated by Masdar, ACWA Power, Total Eren.</li>
      <li><strong>Wind</strong> sees the biggest scenario sensitivity ($7-17 bn). Karakalpakstan corridor is the focus area; 1,450 km of new transmission planned.</li>
      <li><strong>Flexible thermal</strong> (CCGT) needs $3-8 bn for replacement & RE backup — relevant to ILF's traditional advisory practice.</li>
      <li><strong>Grid & T&D</strong> losses still ~9% — modernisation capex implicit but not in chart; ILF EE practice opportunity.</li>
    </ul>
    </div>
  </section>

  <section id='docs' class='tab'>
    <h2>Source documents</h2>
    <p class='lede'>Primary references for this tracker. Capstone-internal files live in <code>readings/</code>; external links open in a new tab.</p>
    <div class='docs-grid'>{docs_html}</div>
  </section>

  <section id='news' class='tab'>
    <h2>Recent news</h2>
    <p class='lede'>Curated developments since April 2024. Snapshot at notebook build time — not auto-updating.</p>
    <div class='card'>{news_html}</div>
  </section>

  <section id='glossary' class='tab'>
    <h2>Glossary — terms used in this dashboard</h2>
    <p class='lede'>Quick reference for non-technical readers. Definitions written for ILF advisory & BD teams.</p>
    <div class='docs-grid' style='grid-template-columns:repeat(auto-fit,minmax(360px,1fr));'>{glossary_html}</div>
  </section>

  <section id='methodology' class='tab'>
    <h2>Methodology in one screen</h2>
    <div class='card'>
      <h3 style='margin-top:0'>Data pipeline</h3>
      <p>Notebook <code>01_data_pipeline.ipynb</code> merges IEA energy balances, IRENA capacity statistics, State Statistics Committee of Uzbekistan (StatSUZ), and World Bank macro indicators into a single annual time series (1990-2024). <em>Bridged</em> series fill IEA-Soviet-era gaps using StatSUZ.</p>
      <h3>Demand forecast</h3>
      <p>Three models trained on 2000-2018 and tested on 2019-2023: ARIMA (grid-searched over <code>(p,d,q)</code> by AIC), additive-trend Holt-Winters, and a log-log GDP-per-capita elasticity regression. The reported ensemble is the simple mean; ARIMA's 80% CI is shown.</p>
      <h3>Generation-mix scenarios</h3>
      <p>Capacity paths interpolate 2024 actuals → 2025 progress → 2030 targets (BAU = 60% of target; Government = official target; Accelerated = stretch). Annual capacity factors: solar 18%, wind 30%, hydro 36%, thermal 55%. Thermal output back-calculated as demand × 1.09 (losses) − RE output.</p>
      <h3>CO₂</h3>
      <p>Power emissions = thermal generation × weighted emission factor. EF<sub>gas</sub> calibrated to match WB historicals (650 gCO₂/kWh, current fleet) and linearly approaches modern-CCGT 380 gCO₂/kWh by 2040 in non-BAU scenarios.</p>
      <h3>Limitations</h3>
      <p>2024 data are preliminary. 2025-2026 actuals from news exceed 2024 by ~10-15% (real demand surge or improved statistics). Forecasts will be revised when official 2025 data is released.</p>
    </div>
  </section>
</main>
</div>
<footer>Built from notebooks <code>01_data_pipeline.ipynb</code> → <code>02_eda_analysis.ipynb</code> → <code>03_forecasting.ipynb</code> → <code>04_dashboard.ipynb</code>. All charts interactive; data CSVs in <code>data/processed/</code>.</footer>
</body>
</html>
"""

out_path = OUT / 'uzbekistan_power_tracker.html'
out_path.write_text(page, encoding='utf-8')
print(f'Wrote → {out_path}')
print(f'Size : {out_path.stat().st_size/1024:.1f} KB')

Wrote → /Users/feya/Downloads/My capstone project/.claude/worktrees/vigorous-matsumoto-672077/outputs/uzbekistan_power_tracker.html
Size : 141.9 KB
